In [0]:
notes_test = spark.read.table("...")
notes_train = spark.read.table("...")

In [0]:
"""
Fine-tune DeBERTa-v3-base on a 6-label multi-label classification task

"""

import os, random, json, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from collections import defaultdict

import torch
from torch.utils.data import DataLoader

from pyspark.sql import SparkSession               
import pandas as pd
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AdamW,
    DataCollatorWithPadding,
)

from sklearn.metrics import precision_recall_fscore_support

spark = SparkSession.builder.getOrCreate()

notes_train_df = notes_train.toPandas()
notes_test_df  = notes_test.toPandas()

LABEL_COLUMNS = [
    "empathic_opportunity",
    "statement_of_emotion",
    "valence_negative",
    "valence_positive",
    "statement_of_progress",
    "statement_of_challenge",
]
NUM_LABELS = len(LABEL_COLUMNS)


# Create Hugging Face Datasets

def make_hf_dataset(pdf: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        pdf[["message_text"] + LABEL_COLUMNS],
        preserve_index=False,
    )

    def _stack_labels(example):
        example["labels"] = [float(example[c]) for c in LABEL_COLUMNS]
        return example

    ds = ds.map(_stack_labels, remove_columns=LABEL_COLUMNS)  
    return ds

hf_train_full = make_hf_dataset(notes_train_df)
hf_test       = make_hf_dataset(notes_test_df)

#Tokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tokenize_function(batch):
    return tokenizer(
        batch["message_text"],
        truncation=True,
        max_length=512,
    )

hf_train_full = hf_train_full.map(tokenize_function, batched=True)
hf_test       = hf_test.map(tokenize_function, batched=True)


hf_train_full.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)
hf_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)


data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

# Metric computation helpers

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()
    labels = labels.astype(int)

    micro_p, micro_r, micro_f, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )
    macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    per_label_p, per_label_r, per_label_f, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    metrics = {
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f,
    }
    for i, lbl in enumerate(LABEL_COLUMNS):
        metrics[f"{lbl}_precision"] = per_label_p[i]
        metrics[f"{lbl}_recall"]    = per_label_r[i]
        metrics[f"{lbl}_f1"]        = per_label_f[i]
    return metrics


# Training + evaluation loop

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
EPOCHS = 4
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.06     # ~6 % of steps
PER_DEVICE_BATCH_SIZE = 4     
GRADIENT_ACCUMULATION = 2

training_sizes = [10, 25, 50, 100, 200, 400, 500]
repetitions    = {s: (1 if s == len(hf_train_full) else 20) for s in training_sizes}


results_store = defaultdict(list)   

for size in training_sizes:
    reps = repetitions[size]
    for rep in range(reps):
        seed = 2025 + rep          
        torch.manual_seed(seed)
        random.seed(seed)

        if size == len(hf_train_full):
            hf_train = hf_train_full
        else:
            sample_indices = random.sample(range(len(hf_train_full)), size)
            hf_train = hf_train_full.select(sample_indices)

        hf_train_tok = hf_train.map(tokenize_function, batched=True)
        hf_train_tok.set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"],
        )

        # build model fresh for every run 
        model = AutoModelForSequenceClassification.from_pretrained(
            "microsoft/deberta-v3-base",
            problem_type="multi_label_classification",
            num_labels=NUM_LABELS,
        ).to(DEVICE)

        #  training arguments 
        training_args = TrainingArguments(
            output_dir=f"./deberta_runs/size{size}_rep{rep}",
            overwrite_output_dir=True,
            evaluation_strategy="no",      
            save_strategy="no",
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION,
            num_train_epochs=EPOCHS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=0.01,
            logging_steps=10,
            seed=seed,
            disable_tqdm=True,
            report_to=[],
            fp16=True,                 
            optim="adamw_torch_fused", 
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=hf_train_tok,
            eval_dataset=hf_test,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        metrics = trainer.evaluate()
       
        for k, v in metrics.items():
            results_store[(size, k)].append(v)

        # free GPU memory for the next run
        del trainer, model
        torch.cuda.empty_cache()



import math, pandas as pd
from tabulate import tabulate
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def mean_ci(values):
    n = len(values)
    mean = sum(values) / n
    if n == 1:
        ci = 0.0
    else:
        sd = math.sqrt(sum((x - mean) ** 2 for x in values) / (n - 1))
        ci = 1.96 * sd / math.sqrt(n)
    return mean, ci

rows = []
all_metrics = sorted({m for (_, m) in results_store.keys()})

for size in training_sizes:
    row = {"train_size": size}
    for metric in all_metrics:
        vals = results_store[(size, metric)]
        mean, ci = mean_ci(vals)
        row[metric] = f"{mean:.4f} ± {ci:.4f}"
    rows.append(row)

pdf_results = pd.DataFrame(rows).set_index("train_size")

results_spark_df = spark.createDataFrame(pdf_results.reset_index())

In [0]:
display(results_spark_df)